# 🔍 한국어 감성 분석 추론 테스트

파인튜닝된 Qwen3-14B + LoRA 어댑터를 로드하여 직접 테스트할 수 있는 노트북입니다.

- **모델**: Qwen/Qwen3-14B (4-bit QLoRA)
- **어댑터 위치**: `outputs/adapter/`
- **출력 형식**: JSON (sentiment, probability, positive_topics, negative_topics)

## 1. 환경 설정 및 모델 로드

In [ ]:
import os
import json
import yaml
import ctypes

# bitsandbytes가 필요로 하는 CUDA 라이브러리를 직접 로드
# (os.environ으로 LD_LIBRARY_PATH를 설정해도 이미 시작된 프로세스에는 효과 없음)
ctypes.cdll.LoadLibrary("/usr/local/lib/python3.11/dist-packages/nvidia/cu13/lib/libnvJitLink.so.13")

import torch
from dotenv import load_dotenv
load_dotenv()

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.0f} GB")

In [ ]:
# 프로젝트 루트 경로 설정 (어디서 실행하든 동작하도록 절대 경로 사용)
PROJECT_ROOT = "/workspace/senti-fine-tuning-test"

# 설정 로드
with open(f"{PROJECT_ROOT}/configs/training_config.yaml") as f:
    config = yaml.safe_load(f)

MODEL_NAME = config["model"]["name"]
ADAPTER_DIR = f"{PROJECT_ROOT}/{config['output']['adapter_dir']}"

print(f"Base 모델: {MODEL_NAME}")
print(f"어댑터 경로: {ADAPTER_DIR}")
print(f"어댑터 파일 존재: {os.path.exists(ADAPTER_DIR)}")

In [ ]:
# 양자화 설정 (학습 시와 동일)
compute_dtype = getattr(torch, config["quantization"]["bnb_4bit_compute_dtype"])
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type=config["quantization"]["bnb_4bit_quant_type"],
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=config["quantization"]["bnb_4bit_use_double_quant"],
)

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 모델 로드 (4-bit 양자화)
print("모델 로딩 중... (1~2분 소요)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",  # PyTorch 내장 attention 사용 (flash_attn 미설치 대응)
)

# LoRA 어댑터 적용
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.eval()

print("모델 로드 완료!")

## 2. 추론 함수 정의

In [ ]:
SYSTEM_PROMPT = "당신은 한국어 감성 분석 전문가입니다. 주어진 텍스트의 감성을 분석하고 결과를 JSON으로 반환하세요."
USER_TEMPLATE = '다음 텍스트의 감성을 분석하세요:\n"{text}"'

def predict(text: str) -> dict:
    """텍스트를 입력받아 감성 분석 결과를 JSON으로 반환"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(text=text)},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=config["inference"]["max_new_tokens"],
            temperature=config["inference"]["temperature"],
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    # JSON 파싱
    try:
        if "{" in response and "}" in response:
            start = response.index("{")
            end = response.rindex("}") + 1
            return json.loads(response[start:end])
    except (json.JSONDecodeError, ValueError):
        pass

    return {"raw_response": response, "error": "JSON 파싱 실패"}


def analyze(text: str):
    """결과를 보기 좋게 출력"""
    print(f"입력: {text}")
    print("-" * 60)
    result = predict(text)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print()
    return result

## 3. 테스트 해보기

아래 셀의 텍스트를 자유롭게 수정하여 테스트해보세요!

In [ ]:
# 긍정 텍스트 테스트
analyze("이 제품 정말 좋아요 향도 좋고 발림성도 최고에요 강추합니다")

In [ ]:
# 부정 텍스트 테스트
analyze("배송도 느리고 포장도 엉망이에요 제품도 기대 이하라 실망입니다")

In [ ]:
# 혼합 감성 테스트
analyze("디자인은 예쁜데 내구성이 별로에요 가격 대비 아쉽습니다")

In [ ]:
# 직접 입력해보세요!
my_text = "여기에 분석할 텍스트를 입력하세요"
analyze(my_text)

## 4. 여러 텍스트 한번에 테스트

In [ ]:
# 여러 텍스트를 리스트로 넣어서 한번에 테스트
texts = [
    "색상이 화면과 똑같아요 너무 만족합니다",
    "사이즈가 생각보다 작아서 교환했어요",
    "가격은 비싸지만 품질이 좋아서 만족해요",
]

results = []
for text in texts:
    result = analyze(text)
    results.append({"text": text, **result})

# 결과 요약 테이블
print("=" * 60)
print("결과 요약")
print("=" * 60)
for r in results:
    sentiment = r.get("sentiment", "N/A")
    prob = r.get("probability", "N/A")
    print(f"  [{sentiment:>8}] (prob: {prob}) {r['text'][:40]}")